In [ ]:
import mne
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy import signal
from autoreject import AutoReject

In [ ]:

def process_subject(file_path):

    # first we erase the data type for the naming
    subject_id = os.path.basename(file_path).replace('.edf', '')

    # === step 1 ===

    # here we read the data from the file path
    # mne.io.read_raw_edf : it opens the subject files
    # preload = true means we load the data into RAM so we can apply filter and modifications to it
    raw = mne.io.read_raw_edf(file_path, preload=True)

    raw.pick_types(eeg=True)
    # here we only keep the brain signals (eeg)  and remove extra channels like triggers or ekg

    # here we extract technical metadata to report
    fs = raw.info['sfreq'] # here we find the sampling frequency

    nyquist = fs / 2 # here we find the nyquist which is the highest frequency the data can show

    n_channels = len(raw.ch_names) # here we find the number of channels : number of electrodes on the head

    duration_sec = raw.n_times / fs # here we find the total lenght of the recording


    print(f"\n{'='*50}")
    print(f"SIGNAL SPECIFICATIONS: {subject_id}")
    print(f"{'-'*50}")
    print(f"Sampling Frequency: {fs} Hz")
    print(f"Nyquist Frequency: {nyquist} Hz")
    print(f"Max Reconstructible Frequency: {nyquist} Hz")
    print(f"Signal Duration: {duration_sec:.2f} seconds")
    print(f"Number of Channels: {n_channels}")
    print(f"{'='*50}\n")

    print("raw signal in the time domain")
    raw.plot(n_channels=n_channels, duration=10, scalings={'eeg': 40e-6}, title='original signal (raw)')
    # here we plot hte signal in the time domain (raw data)
    plt.show()

    # here we design the filter => butterworth filter
    low_cut = 0.5 # here we block the DC drift => caused by moving or breathing , etc.
    high_cut = 45 # here we block the noise => caused by muscle tension or electronic interference
    order = 4 # here we ensure a sharp transition between the frequencies we want to keep and the noise we want to block


    pass_band_filter = signal.butter(order , [low_cut , high_cut] , btype = 'band' , output = 'sos' , fs=fs)
    #creating the band-pass-filter
    # signal.butter designs the mathematical coefficients for the filter
    # btype = band tells it to allow a specific band (0.5 to 45 hz) and block everything else
    # sos break the 4th order  filter into two smaller 2nd filters to ensure numerical precision and stability and preventing errors


    w , h = signal.sosfreqz(pass_band_filter , worN = 2000 , fs=fs)
    # finding the پاسخ فرکانسی
    # signal.sosfreqz calculates the frequency responde

    db_magnitude = 20 * np.log10(np.abs(h) + 1e-20)
    # turning the domain to db we need to 20 * log10(|h|) => convert the magnitude into db and we add a little number to it to prevent the divide by zero

    plt.plot(w , db_magnitude , color = 'blue')
    # ploting the db plot

    plt.axvspan(low_cut , high_cut , color = 'green' , label = "pass-band-filter" , alpha = 0.2)
    # here we add a green area between the threshold, the signals that can pass the filter and we show the thresholds with red dashed lines
    plt.axvline(low_cut , color = 'red' , linestyle='--' , label = "lower cutff")
    plt.axvline(high_cut , color = 'red' , linestyle='--' , label = "upper cutff")
    plt.title('Bode Plot')
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Magnitude (dB)')
    plt.legend()
    plt.show()


    print("Calculating the PSD before the filtering...")
    # psd shows the power of the signal, it tells us which frequencies are the loudest
    # here we plot the psd of the raw data
    fig_pre = raw.compute_psd(fmin=1, fmax=100).plot(picks='eeg', show=False)

    fig_pre.suptitle(f'PSD Before Filtering - {subject_id}', fontsize=12)
    for ax in fig_pre.axes:
        ax.axvline(50, color='red', linestyle='--', alpha=0.6, label='50Hz Noise') # here we add the dashed line at 50hz, to show the noise at this point
    plt.show()

    # filtering
    print("Notch filtering...")
    raw.notch_filter(freqs=50) # here we apply the notch filter and we cut out the 50hz frequencies
    print("Applying pass-band-filtering...")
    raw.filter(l_freq=0.5, h_freq=45.0) # here we remove the low frequencies and high frequencies

    # here we plot the psd fafter the filtering
    print("Calculating the PSD after the filtering...")
    fig_post = raw.compute_psd(fmin=1, fmax=100).plot(picks='eeg', show=False)
    fig_post.suptitle(f'PSD After Filtering (0.5-45Hz) - {subject_id}', fontsize=12)
    for ax in fig_post.axes:
        ax.axvline(50, color='red', linestyle='--', alpha=0.6, label='50Hz Removed')
    plt.show()

    # here we plot the filtered signal in the time domain
    print("filtered signal in the time domain")
    raw.plot(n_channels=10, duration=10, scalings={'eeg': 40e-6},
             title=f'Filtered Signal (Time Domain) - {subject_id}')
    plt.show()




    # === step 2 ===
    # montage
    raw.rename_channels(lambda x: x.replace('-LE', '').upper())
    # here we iterate over all the channel names and  standardize the names , we erase the LE suffix and uppercase all the letters left

    mapping = {ch: ch.capitalize().replace('Z', 'z').replace('Fp', 'Fp') for ch in raw.ch_names}
    raw.rename_channels(mapping)
    # here we make a new dictionary tha store the old and new name of the channels, we capitalize the first letters of the names and then make the extra changes like replacing some letters, so the the names exactly match standard 1020 channel labels

    # here we assign the montage
    montage = mne.channels.make_standard_montage("standard_1020") # here we load the 1020 system electrode positions
    raw.set_montage(montage, on_missing='warn') #here we assign spatial locations to channels(assigning each channel its location), if a channel has no matching location we give a warning instead of crashing


    # here we remove the channels without valid positions => keep only the eeg channels
    # this prevents AutoReject from crashing on channels without positions
    raw.pick_types(eeg=True, selection=None, exclude=[])


    # here we only keep eeg channels that exist in the montage and valid locations => this prevent autoreject from the crashing
    raw.pick([ch for ch in raw.ch_names if ch in montage.ch_names])

    # epoching => segmenting continuous eeg
    epochs = mne.make_fixed_length_epochs(raw, duration=2.0, preload=True)
    # we are splitting the continuous eeg into 2 second, non-overlapping  epoches
    # preload = true => meaning we load the data in the memory

    epochs.set_eeg_reference(ref_channels='average')
    # here we apply average reference => each channel is referenced to the mean of all channels

    epochs.apply_baseline(baseline=(None, None))
    # here we remove the mean of each epoch => helps to stabilize the amplitude

    # autorejecting
    ar = AutoReject(random_state=42, verbose=False)
    # here we are initializing auto reject
    epochs_clean, reject_log = ar.fit_transform(epochs, return_log=True)
    # fit => learns thresholds per channel
    # transform => applies  rejection or interpolated

    # rejected_log.labels : 0 => bad , 1 => good , 2 => interpolated
    custom_cmap = ListedColormap(['#2ecc71', '#e74c3c', '#3498db'])
    # applying the colors for the heatmap
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # here we display the decision matrix : x-axis => channels , y-axis => epochs
    im = ax.imshow(reject_log.labels, aspect='auto', cmap=custom_cmap, interpolation='nearest')
    
    ax.set_title(f"AutoReject Decision Matrix: {subject_id}", fontsize=14, pad=15)
    ax.set_xlabel("EEG Channels", fontsize=12)
    ax.set_ylabel("Epoch Number (2s segments)", fontsize=12)
    
    ax.set_xticks(range(len(epochs.ch_names))) # labels each column with channel name
    ax.set_xticklabels(epochs.ch_names, fontsize=8, rotation=90) #

    # we add a color bar with specific labels
    cbar = plt.colorbar(im, ticks=[0.33, 1, 1.66])
    cbar.ax.set_yticklabels(['Good', 'Bad', 'Interpolated'])
    
    plt.tight_layout()
    plt.show()

    # here we are making a matrix for storing the status
    labels = reject_log.labels
    n_epochs, n_channels = labels.shape

    # here we calculate the status of all the channels
    total_good = np.sum(labels == 0) # find the number of good epochs
    total_bad = np.sum(labels == 1) # find he number of bad epochs
    total_interp = np.sum(labels == 2) # find the number of interpolated epochs
    total_cells = n_epochs * n_channels # find the total cells of the matrix

    print(f"\n{'#'*60}")
    print(f" STATISTICAL OVERALL REPORT: {subject_id}")
    print(f"{'#'*60}")
    print(f"Total Channels analyzed: {n_channels}")
    print(f"Total Epochs analyzed:   {n_epochs}")
    print(f"Total Data Segments:     {total_cells}")
    print(f"{'-'*60}")
    print(f"Total GOOD segments:     {total_good}  ({(total_good/total_cells)*100:.1f}%)")
    print(f"Total BAD segments:      {total_bad}   ({(total_bad/total_cells)*100:.1f}%)")
    print(f"Total INTERPOLATED:      {total_interp} ({(total_interp/total_cells)*100:.1f}%)")
    print(f"{'#'*60}\n")

    # we apply the 70% rule
    bad_counts = (labels > 0).sum(axis=0)
    # count of bad epochs per channel = bad + interpolated
    threshold = 0.7 * n_epochs # here we define he 70% failure threshold
    global_bads = [epochs.ch_names[idx] for idx, count in enumerate(bad_counts) if count > threshold]
    # here add epochs that do not follow the 70% rule

    print(f"DETAILED PER-CHANNEL ANALYSIS:")
    print(f"{'Channel':<10} | {'Bad/Interp':<12} | {'Percentage':<10} | {'Status'}")
    print("-" * 55)
    for idx, ch in enumerate(epochs.ch_names):
        perc = (bad_counts[idx] / n_epochs) * 100
        # here we calculate the percentage of the bad epochs
        status = "HEALTHY" if ch not in global_bads else "!!! GLOBAL BAD !!!"
        print(f"{ch:<10} | {bad_counts[idx]:<12} | {perc:>8.1f}% | {status}")

    print(f"\nFinal Identified Global Bads for Reconstruction: {global_bads}")
    print(f"{'='*60}\n")

    # highlighting the bad channels sensor in red
    epochs.info['bads'] = global_bads
    # here we mark bad channels in MNE metadata
    fig_sensors = epochs.plot_sensors(kind='topomap', show_names=True)
    # here we plot the scalp layout and sensors
    plt.title(f"EEG Sensor Layout & Faulty Channels (Red): {subject_id}")
    plt.show()

    # here we interpolated the bad channels
    # we reconstruct bad channels
    # reset_bad = true => clears bad labels after interpolation
    epochs_clean.interpolate_bads(reset_bads=True)
    
    # here we plot the signal for the comparison => before and after interpolation
    # plotting to show how flat/noisy channels were repaired
    print("signal before the interpolation")
    epochs.plot(n_epochs=2, n_channels=10, title=f"BEFORE: Noisy/Flat Signal - {subject_id}", scalings={'eeg': 40e-6})
    print("signal after the interpolation")
    epochs_clean.plot(n_epochs=2, n_channels=10, title=f"AFTER: Interpolated Signal - {subject_id}", scalings={'eeg': 40e-6})


    #====step 3====
    # here in this step we remove the artifacts and noise using the ICA
    print("removing artifacts using ICA")

    # first we initializing the ica
    ica = mne.preprocessing.ICA(n_components = 0.95 , random_state = 42 , method = 'fastica')
    # n_components => means it will select the number of components
    # that explain the 95% of the variance

    # here we fit the ica to the cleaned epochs
    ica.fit(epochs_clean)

    # here we identify the artifacts like th eye blinks or eye movements ,...
    # we use create_eog_epochs or correlation with frontal channels to find the eye related components => we use fp1 as a proxy
    eog_indices , eog_scores = ica.find_bads_eog(epochs_clean , ch_name = 'Fp1' , threshold = 3)


    print(f"identified eog components : {eog_indices}")
    ica.exclude = eog_indices # we mark these for removal

    # here we plot the components to see their distribution using the topomap
    print("plotting the ica components")
    print("topomap plot")
    ica.plot_components(title = f"ica components - {subject_id}")
    plt.show()


    if eog_indices: # here we plot the properties of the excluded components
        ica.plot_properties(epochs_clean , pick = eog_indices)
        plot.show()

    # here we apply ica => this removes the excluded components for the signal
    print("applying ica to clean the signal")
    epochs_final = ica.apply(epochs_clean.copy())


    print("comparing the raw vs final cleaned signal")
    print("comparison plot")
    epochs_final.plot(n_epochs = 2 , n_channels = 10, title = f"final ica cleaned - {subject_id}")


    
    return epochs_clean , epochs_final

# loop for all subjects
folder_path = r"C:\Users\KaraPardazesh\Desktop\SS PROJECT\subjects"

for i in range(1, 11):
    file_name = f'Subject_{i:02d}.edf'
    full_path = os.path.join(folder_path, file_name)
    
    if os.path.exists(full_path):
        print(=======================================)
        print(f"Starting analysis for {file_name}...")
        print(=======================================)
        processed_data = process_subject(full_path)

before the filtering we have a bump in the 50hz which is for the power line noise(نویز برق شهری ). but after the filtering the bump seems to get flattered and gotten straightened.

for the first step:
after applying the filter we should see these results:
1. with applying the notch filter we omit the 50Hz noise caused by the powerline noiese. in the psd plot before the filtering we can see a peak or sth like a bump in the 50Hz which is caused by the power line noise. by using the notch filter we remove this noise and in the plot after the filtering we can see that that peak at 50Hz is flattened now.
2. by using the band pass filter:
for low frequencies : power is reduced cause we remove the DC drifts caused by non-physical factors like the electrodes movements or sweating.
for high frequencies : here we remove the EMG noises( noises caused by the muscle activities) and thermal noise from the hardware we are using the record the data.

answer to the question: the aliasing
* if our signal contains frequencies higher than the nyquist, aliasing happens. based on the nyquist theory, we can reconstruct a signal perfectly if its sampling rate is twice the highest frequency component that we have in the signal. if this condition is not met, it might cause:
1. higher frequency component get mixed with lower frequency components, which is frequency overlapping.
2. if aliasing happens, then it is impossible to distinguish between the brian signal and alias frequencies, and this would make the analysis hard.

for the second step: answer to the questions
1. difference between a red column and a red row in the heatmap : 
* when a column is completely red means that a sensor failed(global bed channel). it means that a specific electrode was recording garbage data all the time, therefore it is a sensor or hardware issue
* when a row is completely red means that there was a momentary artifact(bad epoch). when a row is red it means that during a specific epoch(2 sec), almost all channels were affected by noise at the same time, like by a movements or....

2. the logic behind the 70% vs 10% rule 
* if a channels is noisy or flat for more of 70% of the recordings the data is unreliable. trying to fix this issue is not good because there is not enough clean and reliable signal left to work with. in this case we discard and ignore the entire channel and use interpolation to reconstruct the channel using its healthy neighboring electrodes and channels.
* is a channels is only bad 10% of time, it means that 90% of the data that has been recorded is reliable. discarding the entire channel would cause a mass loss of information, so instead we only repair that 10% bad epochs and keep the 90% remaining data



* summery of step 2
1. montage and channel mapping: we rename the channels to follow the 10-20 system. this tells the software the exact place of each electrode on the head
2. referencing: here we apply average reference(subtracting the mean of all channels form each channel) to remove common noise and adjust the baseline to the center the signal at zero.
3. autpreject : we use autoreject to automatically scan the data for finding the artifacts and noised. it creates a heatmap to identify bad segments or sensors
4. the 70% rule and interpolation: if a channel was bad for more than 70% of the time, we identify it as a global bad channel. then we use interpolation to repair these bad channels by calculating what they should have recorded based on the activity of their healthy neighbors.


for step 3 : answer to the questions
1. Wyy didn't we use a low-pass or high-pass filter to remove eye blinks?
* while eye blinks have a specific frequency range we can not use filter above to remove them cause:  
1. actual brain waves exist in the same frequency range , if we use a filter to cut that frequency(related to eye blink), we would also destroy the real brain activity happening at that time.
2. a blink is a massive electrical discharge compared to brain activity => meaning its amplitude is much larger. so filtering alone can't clean that big noise without distorting the signal.
3. we use ica because it separated signals based on statistical independence rather that frequency. ica can identify the source of the blink and remove it and leave brain waves untouched, even if the share the same frequency.


2. what does the blink component look like on a Topography map?
* in topography map, the eye blink is very different from the rest of the components. it appears as concentrated hot spots of energy at the top/front edge of the map. the energy is located over fp1 and fp2 electrodes sites.

3. why is its energy concentrated at the front of the head?
* the eyes are located underneath the frontal bone, below the fp1 and fp2. when we blink or more our eys, we create a strong electrical discharge. because the frontal electrodes are the nearest sensors to this electrical sourse, they record most of the magnitude of this noise. as we move to the back of the head, the distance to this source increases and the blink noise becomes weaker.


* summery of step 3: 
1. ica decomposition: we break the eeg signal down to independent components. here we separates brain activity signals from the blink noise.
2. identify the eye artifacts(eog): we use fp1 electrode as a reference to find which components correlated with eye blinks.
3. plotting components: we plot these components on a head map. an eye blink component shows a very high concentration of energy at the very front of the head(near the eyes)
4. reconstruction: once we find the blink component, we mark it for removal. then we reconstruct the signal without these components. as a result, the eye blinks disappear fromm the time domain signal and the brain waves are preserved.